# Biasedness calculation

- Take an expanding window of history. Each iteration adds one `BASE_PERIOD` of extra history.
- Produce forecasts one base-period ahead.
- Store the forecast error `actual - forecast`, keeping every summing-matrix row separately (top level = 1 row, bottom level = `BASE_PERIOD` rows).

After collecting ~30 such rolling errors we run a one-sample Student's t-test **per series and per summing-matrix row**, testing $H_0: \mathbb{E}[\text{error}] = 0$ (unbiasedness) against a two-sided alternative (biasedness). Each series therefore gets one test at the top level and `BASE_PERIOD` tests at the bottom level, each over the 30 rolling samples.

In [ ]:
import pickle
from pathlib import Path
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm

from python.config import load_config

cfg = load_config(os.environ.get('CONFIG', 'config.json'))

import numpy as np
import pandas as pd
from scipy import stats

import python.utils as utils
from python.temporal_hierarchy import TemporalHierarchy
from python.time_series import BASE_FORECASTS, VALIDATION_SET
from python.aggregation import BASE_PERIOD, BOTTOM_LEVEL, TOP_LEVEL
from python.base_forecasting import MODEL_MIN_TRAIN_OBS

In [ ]:
MIN_SERIES_LENGTH = cfg.HORIZON + MODEL_MIN_TRAIN_OBS[cfg.MODEL] * BASE_PERIOD

In [ ]:
LEVELS = [TOP_LEVEL, BOTTOM_LEVEL]

In [ ]:
if cfg.DATASET.NAME == 'web-traffic':
    X_raw = utils.fetch_dataset_web_traffic()
    X = utils.filter_last_days_available(X_raw, 28*100)
if cfg.DATASET.NAME == 'M5':
    X = utils.fetch_dataset_m5()
if cfg.DATASET.NAME == 'M4':
    X = utils.fetch_dataset_m4()

X = utils.filter_last_days_available(X, cfg.HORIZON)
X = utils.reduce_dataset(X, start=cfg.DATASET.START, end=cfg.DATASET.END)
X = utils.align_fill_and_trim_series(X)
X = utils.filter_min_length(X, MIN_SERIES_LENGTH)

In [ ]:
N_ROLLS = 30
STEP_PERIODS = 1
MIN_VALID_ROLLS = 10

total_days = len(X)
total_periods = total_days // BASE_PERIOD

min_train_periods = MODEL_MIN_TRAIN_OBS[cfg.MODEL]

max_rolls = max(0, (total_periods - 1 - min_train_periods) // STEP_PERIODS + 1)
if max_rolls == 0:
    raise ValueError(
        f'Not enough history for even one roll: {total_periods} base-periods '
        f'available but the {cfg.MODEL} model needs {min_train_periods} train + 1 held-out.'
    )

n_rolls = min(N_ROLLS, max_rolls)
if n_rolls < N_ROLLS:
    print(f'WARNING: only {total_periods} base-periods available; '
          f'reducing rolls from {N_ROLLS} to {n_rolls}.')

first_train_periods = total_periods - 1 - (n_rolls - 1) * STEP_PERIODS
cutoffs = [
    (first_train_periods + 1 + i * STEP_PERIODS) * BASE_PERIOD
    for i in range(n_rolls)
]

print(f'{total_periods} base-periods available; collecting {n_rolls} rolling origins.')
print(f'Train window grows from {first_train_periods} to {total_periods - 1} base-periods.')

In [ ]:
level_names = [str(a) for a in LEVELS]
series_columns = X.columns.tolist()
n_series = len(series_columns)

level_width = {str(level): BASE_PERIOD // level.period for level in LEVELS}

errors_by_level = {name: [] for name in level_names}

for cutoff in tqdm.tqdm(cutoffs, desc='Rolling origins'):
    X_window = X.head(cutoff)

    hierarchy = TemporalHierarchy(LEVELS)
    try:
        hierarchy.resample_time_series(X_window)
        for _ in hierarchy.make_base_forecasts(BASE_PERIOD, model=cfg.MODEL,
                                               n_jobs=cfg.N_JOBS, verbose=cfg.VERBOSE):
            pass
    except Exception as e:
        tqdm.tqdm.write(f'Skipping roll (cutoff={cutoff}): {type(e).__name__}: {e}')
        for name in level_names:
            errors_by_level[name].append(np.full((n_series, level_width[name]), np.nan))
        continue

    for level, name in zip(LEVELS, level_names):
        fl = hierarchy[level]
        width = level_width[name]
        try:
            # flat: (n_series, len, width); len == 1 for a single held-out period.
            pred = fl[BASE_FORECASTS].flat[:, 0, :]
            actual = fl[VALIDATION_SET].flat[:, 0, :]
            error = actual - pred
        except Exception as e:
            tqdm.tqdm.write(f'Skipping level {name} at cutoff={cutoff}: '
                            f'{type(e).__name__}: {e}')
            error = np.full((n_series, width), np.nan)
        errors_by_level[name].append(error)

# Stack rolls -> tensor of shape (n_rolls, n_series, width) per level.
error_tensors = {
    name: np.stack(rolls, axis=0)
    for name, rolls in errors_by_level.items()
}

for name, tensor in error_tensors.items():
    print(f'{name}: error tensor {tensor.shape} (rolls x series x summing-rows)')

In [ ]:
ALPHA = 0.05
pvalue_columns = {}

for name, tensor in error_tensors.items():
    n_rolls, n_series, width = tensor.shape

    n_valid = np.isfinite(tensor).sum(axis=0)
    testable = n_valid >= MIN_VALID_ROLLS

    with np.errstate(invalid='ignore'):
        res = stats.ttest_1samp(tensor, popmean=0.0, axis=0, nan_policy='omit')
        p_value = np.asarray(res.pvalue, dtype=float)   # (n_series, width)

    p_value = np.where(testable, p_value, np.nan)

    n_skipped = int((~testable).sum())
    if n_skipped:
        print(f'{name}: skipping {n_skipped}/{n_series * width} cells with '
              f'< {MIN_VALID_ROLLS} valid rolls.')

    for r in range(width):
        col = f'{name}{r}'
        pvalue_columns[col] = p_value[:, r]

# Rows = time-series names, columns = level/summing-row, cells = t-test p-values.
pvalues = pd.DataFrame(pvalue_columns, index=series_columns)
pvalues.index.name = 'series'
pvalues['all_unbiased'] = (pvalues.ge(ALPHA) | pvalues.isna()).all(axis=1)
pvalues

In [ ]:
# Per-column stats over the p-value columns only (exclude the all_unbiased flag).
pval_only = pvalues.drop(columns='all_unbiased')
summary = pd.DataFrame({
    'n_tested': pval_only.notna().sum(),
    'n_biased': (pval_only < ALPHA).sum(),
    'fraction_biased': (pval_only < ALPHA).sum() / pval_only.notna().sum(),
    'n_skipped': pval_only.isna().sum(),
})
summary

In [ ]:
Path(cfg.FOLDER).mkdir(parents=True, exist_ok=True)
with open(f"metrics/{cfg.FOLDER}/biasedness_stats.pkl", "bw") as f:
    pickle.dump(pvalues, f)